In [1]:
from collections import Counter
import pandas as pd
from sklearn.cluster import MeanShift, estimate_bandwidth
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


data = pd.read_excel(r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage4\296×87.xlsx")
X = data.iloc[:, 3:]
X
feature_names = X.columns
X = StandardScaler().fit_transform(X)

min_val = np.min(X, axis=0)
max_val = np.max(X, axis=0)
X = (X - min_val) / (max_val - min_val)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)
X.shape

(296, 87)

In [2]:

def extract_numbers(file_path):
    df = pd.read_excel(file_path)
    valid_numbers = []
    for index, value in df['composition'].items():
        if not pd.isna(value):
            valid_numbers.append(int(df.at[index, 'number']))
    return valid_numbers

file_path = r"E:\desktop\Multiobjective Identification of Phosphor Materials via a Domain Knowledge Integrated Semisupervised Learning Framework\dataset\stage4\benchmark4.xlsx"
reported_structures = extract_numbers(file_path)
print(reported_structures)

for j in range(30, 100, 5):
    j = j/100

    pca = PCA(n_components= j)
    pca_data = pca.fit_transform(X)
    num_pca = pca_data.shape[1]

    for q in np.arange(0.01, 0.1, 0.001):
        q = round(q, 2)
        bandwidth = estimate_bandwidth(pca_data, quantile=q)
        ms = MeanShift(bandwidth=bandwidth, cluster_all=True, bin_seeding=True)
        ms.fit(pca_data)
        labels = ms.labels_
        unique_labels = np.unique(labels)

        if 7 > len(np.unique(labels)) > 1:
            print(f"\nSilhouette score: {silhouette_score(pca_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(pca_data, labels):.2f}")


            data['cluster_label'] = labels
            print(f"\nPCA= {j}:", f"Number of clusters k={len(np.unique(labels))}:",f"\nBandwidth evaluation index q= {q}:")
            dict = {}

            label_1 = []
            for num  in  reported_structures:
                label = labels[num-1]
                label_1.append(int(label))
            print(label_1)

            for label in unique_labels:

                cluster_samples = data[data['cluster_label'] == label]
                total_in_cluster = len(cluster_samples)

                reported_in_cluster = len([num for num in cluster_samples['number']
                                            if num in reported_structures])
                print(f"Cluster {label}: {reported_in_cluster}/{total_in_cluster} "
                        f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


            print("Label distribution:", Counter(labels))
        else:
            pass


[14, 15, 16, 17, 95, 111, 142, 144, 153, 157, 171, 180, 199, 261, 283]

Silhouette score: 0.36 
Calinski-Harabasz score: 189.64

PCA= 0.4: Number of clusters k=5: 
Bandwidth evaluation index q= 0.09:
[1, 1, 1, 0, 1, 0, 0, 3, 0, 0, 1, 0, 2, 1, 1]
Cluster 0: 6/89 (Reported ratio: 6.74%)
Cluster 1: 7/111 (Reported ratio: 6.31%)
Cluster 2: 1/52 (Reported ratio: 1.92%)
Cluster 3: 1/39 (Reported ratio: 2.56%)
Cluster 4: 0/5 (Reported ratio: 0.00%)
Label distribution: Counter({np.int64(1): 111, np.int64(0): 89, np.int64(2): 52, np.int64(3): 39, np.int64(4): 5})

Silhouette score: 0.36 
Calinski-Harabasz score: 189.64

PCA= 0.4: Number of clusters k=5: 
Bandwidth evaluation index q= 0.09:
[1, 1, 1, 0, 1, 0, 0, 3, 0, 0, 1, 0, 2, 1, 1]
Cluster 0: 6/89 (Reported ratio: 6.74%)
Cluster 1: 7/111 (Reported ratio: 6.31%)
Cluster 2: 1/52 (Reported ratio: 1.92%)
Cluster 3: 1/39 (Reported ratio: 2.56%)
Cluster 4: 0/5 (Reported ratio: 0.00%)
Label distribution: Counter({np.int64(1): 111, np.int64(0): 89, 

In [5]:
pca = PCA(n_components= 0.65, random_state=42)
new_data = pca.fit_transform(X)
print(new_data.shape)
ratios = pca.explained_variance_ratio_
components = pca.components_
X = pd.DataFrame(X)

components_df = pd.DataFrame(
    components,
    columns=X.columns,
    index=[f"PC{i+1}" for i in range(components.shape[0])]
)

output_file = "pca_components.xlsx"
components_df.to_excel(output_file, index=True)

bandwidth = estimate_bandwidth(new_data, quantile=0.05)
ms = MeanShift(bandwidth=bandwidth, cluster_all=True, bin_seeding=True)
ms.fit(new_data)
labels = ms.labels_
unique_labels = np.unique(labels)

df = pd.DataFrame(labels)
df.to_excel("Meanshift_result_1.xlsx", index=False)

if len(np.unique(labels)) > 1:
    print(f"\nSilhouette score: {silhouette_score(new_data, labels):.2f}", f"\nCalinski-Harabasz score: {calinski_harabasz_score(new_data, labels):.2f}")
else:
    pass

data['cluster_label'] = labels
print(f"Number of clusters k={len(np.unique(labels))}:", )
dict = {}

label_1 = []
for num  in  reported_structures:
    label = labels[num-1]
    label_1.append(int(label))
print(label_1)

for label in unique_labels:

    cluster_samples = data[data['cluster_label'] == label]
    total_in_cluster = len(cluster_samples)

    reported_in_cluster = len([num for num in cluster_samples['number']
                                     if num in reported_structures])
    print(f"Cluster {label}: {reported_in_cluster}/{total_in_cluster} "
                  f"(Reported ratio: {reported_in_cluster/total_in_cluster:.2%})")


print("Label distribution:",  Counter(labels))

(296, 8)

Silhouette score: 0.21 
Calinski-Harabasz score: 44.15
Number of clusters k=6:
[1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1]
Cluster 0: 7/126 (Reported ratio: 5.56%)
Cluster 1: 8/139 (Reported ratio: 5.76%)
Cluster 2: 0/21 (Reported ratio: 0.00%)
Cluster 3: 0/4 (Reported ratio: 0.00%)
Cluster 4: 0/3 (Reported ratio: 0.00%)
Cluster 5: 0/3 (Reported ratio: 0.00%)
Label distribution: Counter({np.int64(1): 139, np.int64(0): 126, np.int64(2): 21, np.int64(3): 4, np.int64(4): 3, np.int64(5): 3})


In [6]:
import pandas as pd
from sklearn.decomposition import PCA
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import Workbook


wb = Workbook()
ws = wb.active


for r in dataframe_to_rows(components_df, index=True, header=True):
    ws.append(r)


red_bold_font = Font(color="FF0000", bold=True)


for row in range(2, ws.max_row + 1):

    values = [ws.cell(row=row, column=col).value for col in range(2, ws.max_column + 1)]


    values = [0 if v is None else v for v in values]


    abs_values = [abs(v) for v in values]
    top_10_indices = sorted(range(len(abs_values)), key=lambda i: abs_values[i], reverse=True)[:10]


    for idx in top_10_indices:
        ws.cell(row=row, column=idx + 2).font = red_bold_font


output_file = "pca_components_highlighted.xlsx"
wb.save(output_file)

print(f"The component-loading matrix has been saved to {output_file}, with the 10 largest absolute values highlighted.")

The component-loading matrix has been saved to pca_components_highlighted.xlsx, with the 10 largest absolute values highlighted.
